# Aurora Grid: A2A Reference Solution

**Language:** Python 3.10+
**Topics:** A2A protocol, Agent Card, JSON-RPC 2.0, tasks and messages, agent discovery and registries, Strands / LangGraph / LangChain
**Level:** Intermediate

This notebook is the complete, executed solution to the Aurora Grid exercise. It writes every lab file to disk, launches four real HTTP services as separate processes, and drives them over the wire.

Nothing here is simulated. Every response you see below came out of a socket.

| Setting | Value |
| --- | --- |
| `a2a-sdk` | 0.3.26 (pinned by `strands-agents<0.4.0`) |
| `strands-agents` | 1.42.0 |
| `langchain` / `langgraph` | 1.3.11 / 1.2.7 |
| Mode | `AURORA_MOCK=1`, fully offline, no AWS, no spend |
| Ports | registry 9100, locator 9101, dispatcher 9102, notifier 9103 |

## How to run this

### VS Code

1. `python -m venv .venv`, then select `.venv` as the notebook kernel
2. Set credentials with `aws configure` or environment variables, never in code (only needed if you flip to real Bedrock)
3. Run the install cell below
4. Run all cells top to bottom. The teardown cell at the end kills every server it started

### Google Colab

1. Run the install cell below
2. Leave `AURORA_MOCK` at `1`. Colab has no AWS credentials by default
3. Run all cells top to bottom

Servers are launched with `subprocess.Popen`, so they survive across cells and die on teardown.

In [1]:
import importlib.metadata as md
import subprocess
import sys

PACKAGES = ["a2a-sdk", "strands-agents", "langchain", "langgraph", "litellm"]


def installed(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

if any(installed(p) is None for p in PACKAGES):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "strands-agents[a2a]==1.42.0", "strands-agents-tools",
         "langchain==1.3.11", "langgraph==1.2.7", "litellm"],
        check=True,
    )

for pkg in PACKAGES:
    print(f"{pkg:<18} {installed(pkg)}")

a2a-sdk            0.3.26
strands-agents     1.42.0
langchain          1.3.11
langgraph          1.2.7
litellm            1.95.0


### The version fact that determines every line below

`pip install a2a-sdk` on its own gives **1.1.2**, which implements A2A spec 1.0 and replaced every Pydantic type with protobuf. `strands-agents 1.42.0` pins `a2a-sdk<0.4.0`, so the install above resolves to **0.3.26**.

| You will see | On 0.3.26 (this notebook) | On 1.1.x |
| --- | --- | --- |
| Card type | Pydantic `AgentCard` | protobuf `a2a_pb2.AgentCard` |
| Server class | `A2AStarletteApplication(...).build()` | `create_jsonrpc_routes(...)` plus `create_agent_card_routes(...)` |
| `a2a.server.apps` | exists | removed |

When a blog post does not match your code, check the version before you check yourself.

## The three layers

```mermaid
flowchart TB
    subgraph L3["Framework layer: how ONE agent thinks"]
        S["Strands<br/>tool loop"]
        G["LangGraph<br/>state machine"]
        C["LangChain<br/>LCEL chain"]
    end
    subgraph L2["Protocol layer: how agents ASK each other"]
        A["A2A<br/>Agent Card, Message, Task, Artifact"]
    end
    subgraph L1["Transport layer: how bytes MOVE"]
        J["JSON-RPC 2.0 over HTTP"]
    end
    S --> A
    G --> A
    C --> A
    A --> J
```

The framework is the chef. A2A is the menu and the order slip. JSON-RPC is the waiter carrying it. Fire the chef and the menu still works, because the menu never described the kitchen.

MCP is what is in your hands. A2A is who is in the room.

## What gets built

```mermaid
flowchart LR
    CO["Coordinator<br/>(this notebook)"]
    RG["Registry :9100<br/>ASK by tag"]
    L["Fault Locator :9101<br/>Strands"]
    D["Crew Dispatcher :9102<br/>LangGraph"]
    N["Customer Notifier :9103<br/>LangChain"]

    CO -->|"GET /agents?tag="| RG
    RG -.->|"KNOCK /.well-known/agent-card.json"| L
    RG -.->|KNOCK| D
    RG -.->|KNOCK| N
    CO -->|"JSON-RPC message/send"| L
    CO -->|"JSON-RPC message/send"| D
    CO -->|"JSON-RPC message/send"| N
```

In [2]:
import os
import pathlib

LAB = pathlib.Path("aurora_lab")
LAB.mkdir(exist_ok=True)
os.environ["AURORA_MOCK"] = "1"     # offline. Set to "0" for real Bedrock in us-east-1.

print("lab directory:", LAB.resolve())
print("AURORA_MOCK  :", os.environ["AURORA_MOCK"])

lab directory: /home/claude/nb/aurora_lab
AURORA_MOCK  : 1


## Stage 1: three agents, zero protocol

Nothing in this file knows what A2A is. Stage 2 wraps these exact three objects without editing their logic.

| Agent | What it is | Loop? | State? | Why this framework |
| --- | --- | --- | --- | --- |
| Fault Locator | `Agent(model, tools)` | Yes, the model decides when to call the tool | Conversation history | A lookup where the model picks the argument |
| Crew Dispatcher | `StateGraph`, two nodes | No, fixed path | Typed dict, checkpointed | A deterministic sequence you must be able to audit |
| Customer Notifier | `prompt \| model \| parser` | No | None | One shot text transform, no decisions |

### Blank answers in this file

| Blank | Answer | The trap |
| --- | --- | --- |
| 1 | `@tool` | An undecorated function is dropped with a stderr line and no exception |
| 2 | `model=strands_model(...)`, `tools=[lookup_feeder]` | `llm=` and `functions=` are LangChain habits |
| 3 | `Annotated[list[str], add]` | Without a reducer every node **replaces** the list |
| 4 | `(b)`, `(c)`, `(a)` in that order | `add_edge(END, START)` compiles into an unreachable cycle |
| 5 | `(b)` then `(a)` | `JsonOutputParser` on a plain text prompt raises at invoke time |

In [3]:
COMMON = '''"""
Aurora Grid A2A lab, shared setup.

    AURORA_MOCK=1   (default)  no AWS, deterministic canned replies
    AURORA_MOCK=0              real Bedrock, us-east-1
"""

import itertools
import os

MOCK = os.getenv("AURORA_MOCK", "1") == "1"

BEDROCK_MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

PORTS = {"registry": 9100, "locator": 9101, "dispatcher": 9102, "notifier": 9103}
BASE = {name: f"http://127.0.0.1:{port}" for name, port in PORTS.items()}

# Port 9000 is deliberately unused. AgentCore Runtime binds 9000 and the Strands
# A2AServer default is also 9000, so any lab that uses it collides the moment a
# second process starts.


def strands_model(canned: str):
    """Model for a Strands agent. Mock or Bedrock, same call site."""
    if MOCK:
        from strands.models.litellm import LiteLLMModel

        # LiteLLM needs the bedrock/ prefix. Strands BedrockModel does not.
        return LiteLLMModel(
            model_id=f"bedrock/{BEDROCK_MODEL_ID}",
            params={"mock_response": canned},
        )
    from strands.models import BedrockModel

    return BedrockModel(model_id=BEDROCK_MODEL_ID, region_name=AWS_REGION)


def langchain_model(canned: str):
    """Chat model for LangChain and LangGraph. Mock or Bedrock, same call site."""
    if MOCK:
        from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

        return GenericFakeChatModel(messages=itertools.cycle([canned]))
    from langchain_aws import ChatBedrockConverse

    return ChatBedrockConverse(model=BEDROCK_MODEL_ID, region_name=AWS_REGION)


def banner(role: str, port: int) -> None:
    mode = "MOCK (no AWS)" if MOCK else f"BEDROCK {AWS_REGION}"
    print(f"[{role}] listening on http://127.0.0.1:{port}  mode={mode}", flush=True)
'''

(LAB / "aurora_common.py").write_text(COMMON)
print("wrote aurora_common.py")

wrote aurora_common.py


In [4]:
LAB1 = '''"""STAGE 1 - three agents, three frameworks, zero protocol."""

from operator import add
from typing import Annotated, TypedDict

from aurora_common import langchain_model, strands_model

# --------------------------------------------------------------------------
# AGENT 1 - Fault Locator (Strands)
# --------------------------------------------------------------------------
from strands import Agent, tool

FEEDER_MAP = {"maple": "F-114", "clinic": "F-114", "harbour": "F-207", "mill": "F-333"}


@tool                                                          # BLANK 1
def lookup_feeder(landmark: str) -> str:
    """Map a street or landmark to the feeder segment that supplies it."""
    for key, feeder in FEEDER_MAP.items():
        if key in landmark.lower():
            return f"{landmark} is on feeder {feeder}"
    return f"No feeder on record for {landmark}"


locator_agent = Agent(
    name="Fault Locator",
    description="Maps an outage complaint to the feeder segment that supplies it.",
    model=strands_model("Maple Street is on feeder F-114."),   # BLANK 2
    tools=[lookup_feeder],                                     # BLANK 2
)

# --------------------------------------------------------------------------
# AGENT 2 - Crew Dispatcher (LangGraph)
# --------------------------------------------------------------------------
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

PRIORITY_KEYWORDS = ("clinic", "hospital", "water", "school")


class DispatchState(TypedDict):
    request: str
    priority: str
    plan: str
    trace: Annotated[list[str], add]                           # BLANK 3


def triage(state: DispatchState) -> dict:
    hit = any(word in state["request"].lower() for word in PRIORITY_KEYWORDS)
    return {"priority": "P1" if hit else "P3", "trace": ["triage"]}


def assign(state: DispatchState) -> dict:
    crew = "ALPHA-2" if state["priority"] == "P1" else "DELTA-7"
    eta = 45 if state["priority"] == "P1" else 180
    return {"plan": f"Crew {crew}, {state['priority']}, ETA {eta} min", "trace": ["assign"]}


_builder = StateGraph(DispatchState)
_builder.add_node("triage", triage)
_builder.add_node("assign", assign)
_builder.add_edge(START, "triage")                             # BLANK 4 (b)
_builder.add_edge("triage", "assign")                          # BLANK 4 (c)
_builder.add_edge("assign", END)                               # BLANK 4 (a)
dispatcher_graph = _builder.compile(checkpointer=InMemorySaver())

# --------------------------------------------------------------------------
# AGENT 3 - Customer Notifier (LangChain LCEL chain)
# --------------------------------------------------------------------------
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

notifier_chain = (
    ChatPromptTemplate.from_template(
        "Write one SMS under 160 characters for an Aurora Grid customer.\\n"
        "Facts: {facts}\\nNo apology, no filler, state the ETA."
    )
    | langchain_model("Aurora Grid: fault found on feeder F-114. "                # BLANK 5 (b)
                      "Crew ALPHA-2 en route, power back by 15:05.")
    | StrOutputParser()                                                            # BLANK 5 (a)
)
'''

(LAB / "lab1_agents.py").write_text(LAB1)
print("wrote lab1_agents.py")

wrote lab1_agents.py


In [5]:
import sys

sys.path.insert(0, str(LAB.resolve()))

from lab1_agents import dispatcher_graph, locator_agent, notifier_chain

print("--- 1. Fault Locator (Strands) ---")
locator_agent("Lights out on Maple Street near the clinic")

print("\n--- 2. Crew Dispatcher (LangGraph) ---")
out = dispatcher_graph.invoke(
    {"request": "Outage on feeder F-114 near the clinic", "trace": []},
    config={"configurable": {"thread_id": "demo-1"}},
)
print(f"priority={out['priority']}  plan={out['plan']}  trace={out['trace']}")

print("\n--- 3. Customer Notifier (LangChain) ---")
print(notifier_chain.invoke({"facts": "feeder F-114, crew ALPHA-2, ETA 45 min"}))

--- 1. Fault Locator (Strands) ---
Maple Street is on feeder F-114.
--- 2. Crew Dispatcher (LangGraph) ---
priority=P1  plan=Crew ALPHA-2, P1, ETA 45 min  trace=['triage', 'assign']

--- 3. Customer Notifier (LangChain) ---
Aurora Grid: fault found on feeder F-114. Crew ALPHA-2 en route, power back by 15:05.


### Blank 3 proved, not asserted

The cell below builds the same graph twice, once with the reducer and once without. Nothing raises either way. That is the whole problem.

In [6]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


def build(annotated: bool):
    fields = {"trace": Annotated[list, add] if annotated else list}
    State = TypedDict("State", fields)
    g = StateGraph(State)
    g.add_node("a", lambda s: {"trace": ["a"]})
    g.add_node("b", lambda s: {"trace": ["b"]})
    g.add_edge(START, "a")
    g.add_edge("a", "b")
    g.add_edge("b", END)
    return g.compile()


print("with    Annotated[list, add] ->", build(True).invoke({"trace": []})["trace"])
print("without Annotated[list, add] ->", build(False).invoke({"trace": []})["trace"])

# Production note: every LangGraph state field that more than one node writes to
# needs a reducer. Without one the last writer wins and the loss is silent.

with    Annotated[list, add] -> ['a', 'b']
without Annotated[list, add] -> ['b']


## Stage 2: wrap them in A2A

Two ways to become an A2A server.

```mermaid
flowchart TB
    subgraph P1["Path A: the framework ships it"]
        SA["Strands Agent"] --> AS["A2AServer(agent=...)"] --> SRV1["HTTP server, card, task store, all of it"]
    end
    subgraph P2["Path B: you write 20 lines"]
        ANY["Anything callable<br/>graph, chain, plain function"] --> EX["AgentExecutor.execute()"]
        EX --> DRH["DefaultRequestHandler"]
        CARD["AgentCard you hand write"] --> APP["A2AStarletteApplication"]
        DRH --> APP --> SRV2["HTTP server"]
    end
```

Path B is the one that matters. It proves A2A has no opinion about what sits behind the executor. The Customer Notifier has no agent loop at all, and the protocol cannot tell.

### The Task state machine

```mermaid
stateDiagram-v2
    [*] --> submitted
    submitted --> working
    working --> input_required: needs a fact from the caller
    working --> auth_required: needs a credential
    input_required --> working: caller replies WITH taskId
    auth_required --> working
    working --> completed
    working --> failed
    working --> canceled
    submitted --> rejected: refused up front
    completed --> [*]
    failed --> [*]
    canceled --> [*]
    rejected --> [*]
```

Four ways out, two ways to wait.

| Exit | Meaning | Who decided |
| --- | --- | --- |
| `completed` | Work finished | Agent |
| `canceled` | Caller pulled the plug mid flight | Caller |
| `failed` | Agent broke while working | Circumstance |
| `rejected` | Agent refused before starting | Agent, deliberately |

`input-required` and `auth-required` are waits, not errors. A missing feeder ID is a question, not a failure.

### The executor shape, memorised

```
submit    -> only if this task id is new
start_work
  ... do the work ...
requires_input   OR   add_artifact + complete   OR   failed
```

### Blank answers in this file

| Blank | Answer | What breaks silently otherwise |
| --- | --- | --- |
| 6 | `enable_a2a_compliant_streaming=True` | Default `False` leaks one history message per stream chunk |
| 7 | `tags=["outage", "locate"]` | Auto-derived skills carry `tags=[]` and no registry query ever matches |
| 8 | `if not context.current_task: await updater.submit()` | Re-submitting a task the store already tracks |
| 9 | `await updater.requires_input(...)` | `failed` is terminal, so the caller retries from scratch |
| 10 | `add_artifact` then `complete` | Completing first closes the task and the artifact lands nowhere |
| 11 | `url=f"{BASE[role]}/"` | A card that only works from your own machine |

In [7]:
LAB2 = '''"""STAGE 2 - put the same three agents behind A2A.

    python lab2_serve.py locator | dispatcher | notifier
"""

import sys

from aurora_common import BASE, PORTS, banner
from lab1_agents import dispatcher_graph, locator_agent, notifier_chain

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.types import AgentCapabilities, AgentCard, AgentSkill


# ==========================================================================
# ROLE A - Fault Locator. Strands ships the whole A2A server.
# ==========================================================================
def serve_locator() -> None:
    from strands.multiagent.a2a import A2AServer

    server = A2AServer(
        agent=locator_agent,
        host="127.0.0.1",
        port=PORTS["locator"],
        version="1.0.0",
        enable_a2a_compliant_streaming=True,                       # BLANK 6
        skills=[
            AgentSkill(
                id="locate_fault",
                name="Locate fault",
                description="Map an outage complaint to the feeder segment that supplies it.",
                tags=["outage", "locate"],                         # BLANK 7
                examples=["Lights out on Maple Street"],
            )
        ],
    )
    banner("locator", PORTS["locator"])
    server.serve()


# ==========================================================================
# ROLE B - Crew Dispatcher. Hand written executor over a LangGraph graph.
# ==========================================================================
class DispatcherExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)

        if not context.current_task:                               # BLANK 8
            await updater.submit()
        await updater.start_work()

        text = context.get_user_input()

        if "f-" not in text.lower():
            await updater.requires_input(                           # BLANK 9
                updater.new_agent_message(
                    [{"kind": "text",
                      "text": "Which feeder segment? Reply with the feeder ID, e.g. F-114."}]
                )
            )
            return

        result = await dispatcher_graph.ainvoke(
            {"request": text, "trace": []},
            config={"configurable": {"thread_id": context.context_id}},
        )
        await updater.add_artifact(                                 # BLANK 10 (b)
            [{"kind": "text", "text": result["plan"]}], name="dispatch_plan"
        )
        await updater.complete()                                    # BLANK 10 (a)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise Exception("Crew Dispatcher does not support cancellation")


# ==========================================================================
# ROLE C - Customer Notifier. Same shape, an LCEL chain behind it.
# ==========================================================================
class NotifierExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        if not context.current_task:
            await updater.submit()
        await updater.start_work()

        sms = await notifier_chain.ainvoke({"facts": context.get_user_input()})
        await updater.add_artifact([{"kind": "text", "text": sms}], name="customer_sms")
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise Exception("Customer Notifier does not support cancellation")


def build_card(role: str, name: str, description: str, skill: AgentSkill) -> AgentCard:
    return AgentCard(
        name=name,
        description=description,
        url=f"{BASE[role]}/",                                       # BLANK 11
        version="1.0.0",
        capabilities=AgentCapabilities(streaming=False),
        default_input_modes=["text"],
        default_output_modes=["text"],
        skills=[skill],
    )


def serve_executor(role: str, card: AgentCard, executor: AgentExecutor) -> None:
    import uvicorn

    handler = DefaultRequestHandler(agent_executor=executor, task_store=InMemoryTaskStore())
    app = A2AStarletteApplication(agent_card=card, http_handler=handler).build()
    banner(role, PORTS[role])
    uvicorn.run(app, host="127.0.0.1", port=PORTS[role], log_level="warning")


CARDS = {
    "dispatcher": (
        "Crew Dispatcher",
        "Assigns a field crew and priority to a confirmed feeder segment.",
        AgentSkill(id="dispatch_crew", name="Dispatch crew",
                   description="Assign a crew and priority to a feeder segment.",
                   tags=["outage", "dispatch"],
                   examples=["Dispatch to feeder F-114 near the clinic"]),
    ),
    "notifier": (
        "Customer Notifier",
        "Turns a dispatch plan into a customer SMS under 160 characters.",
        AgentSkill(id="draft_notice", name="Draft outage notice",
                   description="Write a customer facing SMS from a dispatch plan.",
                   tags=["outage", "comms"],
                   examples=["feeder F-114, crew ALPHA-2, ETA 45 min"]),
    ),
}

EXECUTORS = {"dispatcher": DispatcherExecutor, "notifier": NotifierExecutor}

if __name__ == "__main__":
    role = sys.argv[1] if len(sys.argv) > 1 else "locator"
    if role == "locator":
        serve_locator()
    elif role in EXECUTORS:
        name, desc, skill = CARDS[role]
        serve_executor(role, build_card(role, name, desc, skill), EXECUTORS[role]())
    else:
        print(f"unknown role: {role}")
        sys.exit(1)
'''

(LAB / "lab2_serve.py").write_text(LAB2)
print("wrote lab2_serve.py")

wrote lab2_serve.py


## Stage 3: the door the spec refuses to build

The A2A spec names three discovery strategies and standardises exactly one.

```mermaid
flowchart TB
    Q["I need an agent"] --> D1{"Do I know its domain?"}
    D1 -->|yes| K["KNOCK<br/>GET /.well-known/agent-card.json<br/>STANDARDISED, RFC 8615"]
    D1 -->|no| D2{"Do I know the capability?"}
    D2 -->|yes| A["ASK<br/>query a registry by skill tag<br/>NOT standardised, you build it"]
    D2 -->|no| DI["DIAL<br/>hardcoded URL, env var, config file<br/>nothing to standardise"]
```

KNOCK, ASK, DIAL. Only the first has a spec behind it.

The registry below populates itself by KNOCKing on a DIALed seed list. Someone always has to know the first address. There is no discovery without a bootstrap.

Because the spec defines no registry API, **your registry is your governance surface.** Skill attestation, prompt injection scanning, tenant scoping, and card signature verification all live here, because there is nowhere else for them to live.

### Blank answers in this file

| Blank | Answer | The trap |
| --- | --- | --- |
| 12 | `A2ACardResolver(...).get_agent_card()` | There is no `agent/getCard` JSON-RPC method. The card is a plain GET |
| 13 | `if tag in skill.get("tags", [])` | Matching against `card["name"]` reviews fine and matches nothing |
| 14 | `(b)`, `(c)`, `(a)` | `Route("/message/send", ...)` confuses a JSON-RPC method name with an HTTP path |

In [8]:
LAB3 = '''"""STAGE 3 - the registry the A2A spec deliberately does not define."""

import httpx
import uvicorn
from starlette.applications import Starlette
from starlette.responses import JSONResponse
from starlette.routing import Route

from aurora_common import BASE, PORTS, banner

from a2a.client import A2ACardResolver

SEEDS = [BASE["locator"], BASE["dispatcher"], BASE["notifier"]]   # DIAL, one level down
CATALOG: dict[str, dict] = {}


async def refresh(request):
    """KNOCK on every seed, cache what answers. Dead agents drop out."""
    CATALOG.clear()
    errors = []
    async with httpx.AsyncClient(timeout=10) as hx:
        for base in SEEDS:
            try:
                card = await A2ACardResolver(                       # BLANK 12
                    httpx_client=hx, base_url=base
                ).get_agent_card()
                CATALOG[card.name] = card.model_dump(exclude_none=True, by_alias=True)
            except Exception as exc:
                errors.append({"base": base, "error": type(exc).__name__})
    return JSONResponse({"registered": sorted(CATALOG), "unreachable": errors})


async def agents(request):
    """ASK. Filter by skill tag, the query a domain name cannot answer."""
    tag = request.query_params.get("tag")
    if not tag:
        return JSONResponse(list(CATALOG.values()))
    hits = [
        card
        for card in CATALOG.values()
        for skill in card["skills"]
        if tag in skill.get("tags", [])                            # BLANK 13
    ]
    return JSONResponse(hits)


async def registry_card(request):
    """The registry publishes its own card, so clients bootstrap with one URL."""
    return JSONResponse({
        "name": "Aurora Registry",
        "description": "Catalog of Aurora Grid outage agents, queryable by skill tag.",
        "url": f"{BASE['registry']}/",
        "version": "1.0.0",
        "protocolVersion": "0.3.0",
        "capabilities": {"streaming": False},
        "defaultInputModes": ["text"],
        "defaultOutputModes": ["text"],
        "skills": [{
            "id": "find_agents",
            "name": "Find agents",
            "description": "GET /agents?tag=<tag> returns matching agent cards.",
            "tags": ["registry", "discovery"],
        }],
    })


app = Starlette(routes=[
    Route("/.well-known/agent-card.json", registry_card),          # BLANK 14 (b)
    Route("/refresh", refresh),                                    # BLANK 14 (c)
    Route("/agents", agents),                                      # BLANK 14 (a)
])

if __name__ == "__main__":
    banner("registry", PORTS["registry"])
    uvicorn.run(app, host="127.0.0.1", port=PORTS["registry"], log_level="warning")
'''

(LAB / "lab3_registry.py").write_text(LAB3)
print("wrote lab3_registry.py")

wrote lab3_registry.py


## Launch the four services

Four processes, four ports. `wait_for_card` polls the well known path until each answers, so the notebook never races a cold server.

In [9]:
import atexit
import subprocess
import sys
import time

import httpx

PROCS = {}
PORTS = {"registry": 9100, "locator": 9101, "dispatcher": 9102, "notifier": 9103}
BASE = {name: f"http://127.0.0.1:{port}" for name, port in PORTS.items()}


def start(role: str):
    script = "lab3_registry.py" if role == "registry" else "lab2_serve.py"
    args = [sys.executable, script] + ([] if role == "registry" else [role])
    PROCS[role] = subprocess.Popen(
        args, cwd=str(LAB), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )


def stop_all():
    for role, proc in PROCS.items():
        if proc.poll() is None:
            proc.terminate()
    for proc in PROCS.values():
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()


atexit.register(stop_all)


def wait_for_card(role: str, timeout: float = 90.0) -> bool:
    """Poll the well known path until the server answers or the clock runs out."""
    url = f"{BASE[role]}/.well-known/agent-card.json"
    deadline = time.time() + timeout
    while time.time() < deadline:
        if PROCS[role].poll() is not None:
            print(f"[{role}] DIED:\n{PROCS[role].stdout.read()[-1500:]}")
            return False
        try:
            if httpx.get(url, timeout=3).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(1.0)
    return False


for role in ["locator", "dispatcher", "notifier", "registry"]:
    start(role)

for role in ["locator", "dispatcher", "notifier", "registry"]:
    ok = wait_for_card(role)
    print(f"{role:<11} pid={PROCS[role].pid:<7} card={'UP' if ok else 'FAILED'}")

# Production note: subprocess.Popen is a notebook convenience. In production each
# of these is its own container with its own scaling, IAM role, and rollout.

locator     pid=601     card=UP
dispatcher  pid=602     card=UP
notifier    pid=603     card=UP
registry    pid=604     card=UP


## KNOCK: read every card

The card is a plain HTTP GET on a well known path. No JSON-RPC is involved.

In [10]:
import json

for role in ["locator", "dispatcher", "notifier"]:
    card = httpx.get(f"{BASE[role]}/.well-known/agent-card.json", timeout=10).json()
    skills = [(s["id"], s.get("tags", [])) for s in card["skills"]]
    print(f"{card['name']:<20} v{card['version']}  proto={card['protocolVersion']}  "
          f"transport={card['preferredTransport']}")
    print(f"{'':<20} url={card['url']}")
    print(f"{'':<20} skills={skills}")
    print(f"{'':<20} streaming={card['capabilities'].get('streaming')}\n")

print("--- the locator card in full ---")
print(json.dumps(httpx.get(f"{BASE['locator']}/.well-known/agent-card.json").json(), indent=2))

Fault Locator        v1.0.0  proto=0.3.0  transport=JSONRPC
                     url=http://127.0.0.1:9101/
                     skills=[('locate_fault', ['outage', 'locate'])]
                     streaming=True

Crew Dispatcher      v1.0.0  proto=0.3.0  transport=JSONRPC
                     url=http://127.0.0.1:9102/
                     skills=[('dispatch_crew', ['outage', 'dispatch'])]
                     streaming=False



Customer Notifier    v1.0.0  proto=0.3.0  transport=JSONRPC
                     url=http://127.0.0.1:9103/
                     skills=[('draft_notice', ['outage', 'comms'])]
                     streaming=False

--- the locator card in full ---
{
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "Maps an outage complaint to the feeder segment that supplies it.",
  "name": "Fault Locator",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Map an outage complaint to the feeder segment that supplies it.",
      "examples": [
        "Lights out on Maple Street"
      ],
      "id": "locate_fault",
      "name": "Locate fault",
      "tags": [
        "outage",
        "locate"
      ]
    }
  ],
  "url": "http://127.0.0.1:9101/",
  "version": "1.0.0"
}


### The retired path still answers

`/.well-known/agent.json` was the path before the spec settled on `agent-card.json`. It still returns 200 and logs a deprecation warning server side. Do not build on it.

In [11]:
for path in ["/.well-known/agent-card.json", "/.well-known/agent.json", "/.well-known/nope.json"]:
    resp = httpx.get(f"{BASE['locator']}{path}", timeout=10)
    print(f"{path:<36} -> HTTP {resp.status_code}")

/.well-known/agent-card.json         -> HTTP 200
/.well-known/agent.json              -> HTTP 200
/.well-known/nope.json               -> HTTP 404


## The wire: raw JSON-RPC, no SDK

Everything the SDKs do is this.

**Out: four keys. Never more, never fewer.**

| Key | Carries |
| --- | --- |
| `jsonrpc` | the stamp, always `"2.0"` |
| `id` | the return address, echoed back |
| `method` | the verb |
| `params` | the parcel |

**Back: `jsonrpc`, `id`, and exactly one of `result` or `error`. Never both.**

In [12]:
DIS = BASE["dispatcher"]


def rpc(method: str, params: dict, req_id: str = "nb-1", url: str = DIS) -> dict:
    """One JSON-RPC call, hand rolled. This is the entire client."""
    body = {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params}
    return httpx.post(url, json=body, timeout=120).json()


turn1 = rpc("message/send", {
    "message": {
        "kind": "message",
        "messageId": "m-1",
        "role": "user",
        "parts": [{"kind": "text", "text": "Send a crew, it is dark near the clinic"}],
    }
})
print(json.dumps(turn1, indent=2)[:1400])

{
  "id": "nb-1",
  "jsonrpc": "2.0",
  "result": {
    "contextId": "41a81272-381e-44bd-bdad-7ebb35879301",
    "history": [
      {
        "contextId": "41a81272-381e-44bd-bdad-7ebb35879301",
        "kind": "message",
        "messageId": "m-1",
        "parts": [
          {
            "kind": "text",
            "text": "Send a crew, it is dark near the clinic"
          }
        ],
        "role": "user",
        "taskId": "a5ff72f4-9b45-4550-9d68-85c3b761cc54"
      }
    ],
    "id": "a5ff72f4-9b45-4550-9d68-85c3b761cc54",
    "kind": "task",
    "status": {
      "message": {
        "contextId": "41a81272-381e-44bd-bdad-7ebb35879301",
        "kind": "message",
        "messageId": "2c89ff22-36a8-4085-b7da-dc5608e45783",
        "parts": [
          {
            "kind": "text",
            "text": "Which feeder segment? Reply with the feeder ID, e.g. F-114."
          }
        ],
        "role": "agent",
        "taskId": "a5ff72f4-9b45-4550-9d68-85c3b761cc54"
      },
 

In [13]:
result = turn1["result"]
TASK_ID = result["id"]
CTX_ID = result["contextId"]

print("kind      :", result["kind"])
print("state     :", result["status"]["state"])
print("taskId    :", TASK_ID)
print("contextId :", CTX_ID)
print("question  :", result["status"]["message"]["parts"][0]["text"])
print("artifacts :", result.get("artifacts", "none yet"))

kind      : task
state     : input-required
taskId    : a5ff72f4-9b45-4550-9d68-85c3b761cc54
contextId : 41a81272-381e-44bd-bdad-7ebb35879301
question  : Which feeder segment? Reply with the feeder ID, e.g. F-114.
artifacts : none yet


### The single most expensive mistake in A2A

The two calls below send **the same text** to **the same agent**. One resumes the ticket. One silently opens a new one and orphans the agent's question.

Both return HTTP 200. Neither logs a warning.

In [14]:
without = rpc("message/send", {
    "message": {"kind": "message", "messageId": "m-2a", "role": "user",
                "parts": [{"kind": "text", "text": "Feeder F-114, clinic block"}]}
}, req_id="nb-2a")["result"]

with_ids = rpc("message/send", {
    "message": {"kind": "message", "messageId": "m-2b", "role": "user",
                "taskId": TASK_ID, "contextId": CTX_ID,
                "parts": [{"kind": "text", "text": "Feeder F-114, clinic block"}]}
}, req_id="nb-2b")["result"]

print(f"original task        : {TASK_ID}")
print(f"WITHOUT taskId       : {without['id']}  state={without['status']['state']}")
print(f"WITH taskId+contextId: {with_ids['id']}  state={with_ids['status']['state']}")
print()
print("resumed?", with_ids["id"] == TASK_ID)
print("orphaned?", without["id"] != TASK_ID)
print("plan:", [p["text"] for a in with_ids.get("artifacts", []) for p in a["parts"]])

original task        : a5ff72f4-9b45-4550-9d68-85c3b761cc54
WITHOUT taskId       : 38985569-ca56-4746-b9b9-139cd8e95e34  state=completed
WITH taskId+contextId: a5ff72f4-9b45-4550-9d68-85c3b761cc54  state=completed

resumed? True
orphaned? True
plan: ['Crew ALPHA-2, P1, ETA 45 min']


### `task_id` alone is enough, and you should still send both

Measured below: the server backfills `contextId` from the stored task. Send both anyway, because `contextId` groups related tasks into one conversation, and the moment you fan out to a second agent that server has no stored task to backfill from.

In [15]:
seed = rpc("message/send", {
    "message": {"kind": "message", "messageId": "m-3", "role": "user",
                "parts": [{"kind": "text", "text": "Crew needed, power out"}]}
}, req_id="nb-3")["result"]

only_task = rpc("message/send", {
    "message": {"kind": "message", "messageId": "m-3b", "role": "user",
                "taskId": seed["id"],
                "parts": [{"kind": "text", "text": "Feeder F-207"}]}
}, req_id="nb-3b")["result"]

print("same task id     :", only_task["id"] == seed["id"])
print("contextId kept   :", only_task["contextId"] == seed["contextId"])
print("state            :", only_task["status"]["state"])

same task id     : True
contextId kept   : True
state            : completed


### `tasks/get`: polling, the thing push notifications exist to replace

Note what is and is not in `history`. The final answer lives in `artifacts`, never in `history`.

In [16]:
fetched = rpc("tasks/get", {"id": TASK_ID}, req_id="nb-4")["result"]

print("state          :", fetched["status"]["state"])
print("history length :", len(fetched.get("history", [])))
for i, msg in enumerate(fetched.get("history", []), 1):
    text = " ".join(p["text"] for p in msg["parts"])
    print(f"  {i}. {msg['role']:<6} {text[:64]}")
print("artifacts      :", [(a["name"], [p["text"] for p in a["parts"]])
                           for a in fetched.get("artifacts", [])])

state          : completed
history length : 3
  1. user   Send a crew, it is dark near the clinic
  2. agent  Which feeder segment? Reply with the feeder ID, e.g. F-114.
  3. user   Feeder F-114, clinic block
artifacts      : [('dispatch_plan', ['Crew ALPHA-2, P1, ETA 45 min'])]


## Error codes: two bands, two different owners

```mermaid
flowchart LR
    E["error.code"] --> B1{"which band?"}
    B1 -->|"-32700 to -32603"| T["The ENVELOPE was wrong.<br/>JSON-RPC's own codes."]
    B1 -->|"-32001 to -32007"| A["The envelope was FINE.<br/>The ask was wrong.<br/>A2A's own codes."]
```

Minus 326xx is the envelope. Minus 320xx is the ask.

In [17]:
probes = [
    ("typo in the method name",  "message/sned", {}),
    ("task id never issued",     "tasks/get",    {"id": "does-not-exist"}),
    ("params that fail schema",  "message/send", {"message": {"role": "user"}}),
    ("cancel a terminal task",   "tasks/cancel", {"id": TASK_ID}),
]

print(f"{'probe':<28} {'code':>7}  message")
print("-" * 78)
for label, method, params in probes:
    err = rpc(method, params, req_id="err").get("error", {})
    print(f"{label:<28} {err.get('code'):>7}  {err.get('message')}")

print()
print("Malformed JSON never reaches the method dispatcher at all:")
raw = httpx.post(DIS, content='{"jsonrpc":"2.0","id":"x","method":', 
                 headers={"Content-Type": "application/json"}, timeout=30).json()
print(f"{'truncated body':<28} {raw['error']['code']:>7}  {raw['error']['message']}")

probe                           code  message
------------------------------------------------------------------------------
typo in the method name       -32601  Method not found
task id never issued          -32001  Task not found


params that fail schema       -32602  Invalid parameters
cancel a terminal task        -32002  Task cannot be canceled - current state: TaskState.completed

Malformed JSON never reaches the method dispatcher at all:


truncated body                -32700  Expecting value: line 1 column 36 (char 35)


In [18]:
from a2a.types import (
    AuthenticatedExtendedCardNotConfiguredError, ContentTypeNotSupportedError,
    InternalError, InvalidAgentResponseError, InvalidParamsError, InvalidRequestError,
    JSONParseError, MethodNotFoundError, PushNotificationNotSupportedError,
    TaskNotCancelableError, TaskNotFoundError, UnsupportedOperationError,
)

print(f"{'code':>7}  {'band':<10}  name")
print("-" * 70)
for cls in [JSONParseError, InvalidRequestError, MethodNotFoundError, InvalidParamsError,
            InternalError, TaskNotFoundError, TaskNotCancelableError,
            PushNotificationNotSupportedError, UnsupportedOperationError,
            ContentTypeNotSupportedError, InvalidAgentResponseError,
            AuthenticatedExtendedCardNotConfiguredError]:
    code = cls.model_fields["code"].default
    band = "envelope" if code <= -32600 else "the ask"
    print(f"{code:>7}  {band:<10}  {cls.__name__}")

   code  band        name
----------------------------------------------------------------------
 -32700  envelope    JSONParseError
 -32600  envelope    InvalidRequestError
 -32601  envelope    MethodNotFoundError
 -32602  envelope    InvalidParamsError
 -32603  envelope    InternalError
 -32001  the ask     TaskNotFoundError
 -32002  the ask     TaskNotCancelableError
 -32003  the ask     PushNotificationNotSupportedError
 -32004  the ask     UnsupportedOperationError
 -32005  the ask     ContentTypeNotSupportedError
 -32006  the ask     InvalidAgentResponseError
 -32007  the ask     AuthenticatedExtendedCardNotConfiguredError


In [19]:
from a2a.types import Role, TaskState, TransportProtocol
from a2a.utils import (
    AGENT_CARD_WELL_KNOWN_PATH, DEFAULT_RPC_URL, EXTENDED_AGENT_CARD_PATH,
    PREV_AGENT_CARD_WELL_KNOWN_PATH,
)

TERMINAL = {"completed", "canceled", "failed", "rejected"}
WAITING = {"input-required", "auth-required"}

print("card path      :", AGENT_CARD_WELL_KNOWN_PATH)
print("retired path   :", PREV_AGENT_CARD_WELL_KNOWN_PATH)
print("extended card  :", EXTENDED_AGENT_CARD_PATH, " (method: agent/getAuthenticatedExtendedCard)")
print("default rpc url:", DEFAULT_RPC_URL)
print("transports     :", [t.value for t in TransportProtocol])
print("roles          :", [r.value for r in Role])
print()
print(f"{'state':<16} {'kind':<10}")
print("-" * 28)
for state in TaskState:
    kind = "terminal" if state.value in TERMINAL else ("waiting" if state.value in WAITING else "active")
    print(f"{state.value:<16} {kind:<10}")

card path      : /.well-known/agent-card.json
retired path   : /.well-known/agent.json
extended card  : /agent/authenticatedExtendedCard  (method: agent/getAuthenticatedExtendedCard)
default rpc url: /
transports     : ['JSONRPC', 'GRPC', 'HTTP+JSON']
roles          : ['agent', 'user']

state            kind      
----------------------------
submitted        active    
working          active    
input-required   waiting   
completed        terminal  
canceled         terminal  
failed           terminal  
rejected         terminal  
auth-required    waiting   
unknown          active    


## Stage 4: the coordinator

No model, no tools, no prompt. Pure protocol.

```mermaid
sequenceDiagram
    autonumber
    participant CO as Coordinator
    participant RG as Registry
    participant L as Locator
    participant D as Dispatcher
    participant N as Notifier

    CO->>RG: GET /agents?tag=locate
    RG-->>CO: Fault Locator card
    CO->>L: message/send "Lights out on Maple Street"
    L-->>CO: Task completed, artifact "feeder F-114"

    CO->>RG: GET /agents?tag=dispatch
    RG-->>CO: Crew Dispatcher card
    CO->>D: message/send "Send a crew, it is dark"
    D-->>CO: Task input-required, "Which feeder segment?"
    CO->>D: message/send + taskId + contextId
    D-->>CO: Task completed, artifact "Crew ALPHA-2, P1, ETA 45"

    CO->>RG: GET /agents?tag=comms
    RG-->>CO: Customer Notifier card
    CO->>N: message/send
    N-->>CO: Task completed, artifact SMS
```

### Where the answer lives

| Task state | Read the text from | Field path |
| --- | --- | --- |
| `completed` | The artifact | `task.artifacts[].parts[].root.text` |
| `input-required` | The status message | `task.status.message.parts[].root.text` |

Two places. A client that checks only one prints nothing on half its traffic.

### Blank answers in this file

| Blank | Answer | The trap |
| --- | --- | --- |
| 15 | `hx.get(.../agents, params={"tag": tag})` | KNOCKing the registry's own card returns the registry, not its catalog |
| 16 | `separator = ""` | Artifact parts are appended, never separated |
| 17 | `msg.task_id` and `msg.context_id` | `msg.message_id = task.id` assigns a real value to the wrong field |
| 18 | `[TransportProtocol.jsonrpc]` | The enum value is `"JSONRPC"`, not the string `"JSON-RPC"` |

In [20]:
resp = httpx.get(f"{BASE['registry']}/refresh", timeout=60).json()
print("registered :", resp["registered"])
print("unreachable:", resp["unreachable"])

print("\n--- ASK by tag ---")
for tag in ["locate", "dispatch", "comms", "billing"]:
    hits = httpx.get(f"{BASE['registry']}/agents", params={"tag": tag}, timeout=30).json()
    print(f"  tag={tag:<10} -> {[c['name'] for c in hits] or 'NOTHING REGISTERED'}")

registered : ['Crew Dispatcher', 'Customer Notifier', 'Fault Locator']
unreachable: []

--- ASK by tag ---
  tag=locate     -> ['Fault Locator']
  tag=dispatch   -> ['Crew Dispatcher']


  tag=comms      -> ['Customer Notifier']


  tag=billing    -> NOTHING REGISTERED


In [21]:
import asyncio

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard, Message, Role, TextPart, TransportProtocol


async def find_by_tag(hx, tag: str) -> AgentCard:
    """ASK door. Capability in, address out."""
    resp = await hx.get(f"{BASE['registry']}/agents", params={"tag": tag})   # BLANK 15
    resp.raise_for_status()
    hits = resp.json()
    if not hits:
        raise LookupError(f"no agent registered for tag '{tag}'")
    return AgentCard.model_validate(hits[0])


def text_of(obj) -> str:
    chunks = [
        p.root.text
        for a in (getattr(obj, "artifacts", None) or [])
        for p in a.parts
        if isinstance(p.root, TextPart)
    ]
    if chunks:
        separator = ""                                                       # BLANK 16
        return separator.join(chunks)
    status_msg = getattr(getattr(obj, "status", None), "message", None)
    if status_msg:
        return " ".join(p.root.text for p in status_msg.parts if isinstance(p.root, TextPart))
    if hasattr(obj, "parts"):
        return " ".join(p.root.text for p in obj.parts if isinstance(p.root, TextPart))
    return ""


async def call(factory, card: AgentCard, message: Message):
    client = factory.create(card)
    last = None
    async for event in client.send_message(message):
        last = event[0] if isinstance(event, tuple) else event
    return last


def followup(text: str, task) -> Message:
    msg = create_text_message_object(Role.user, text)
    msg.task_id = task.id                                                    # BLANK 17 (a)
    msg.context_id = task.context_id                                         # BLANK 17 (b)
    return msg


async def outage_run():
    async with httpx.AsyncClient(timeout=120) as hx:
        await hx.get(f"{BASE['registry']}/refresh")
        factory = ClientFactory(ClientConfig(
            httpx_client=hx,
            streaming=False,
            supported_transports=[TransportProtocol.jsonrpc],                # BLANK 18
        ))

        locator = await find_by_tag(hx, "locate")
        print(f"[ASK] tag=locate     -> {locator.name} @ {locator.url}")
        found = await call(factory, locator, create_text_message_object(
            Role.user, "Lights out on Maple Street near the clinic"))
        print(f"  locator says: {text_of(found)!r}\n")

        dispatcher = await find_by_tag(hx, "dispatch")
        print(f"[ASK] tag=dispatch   -> {dispatcher.name} @ {dispatcher.url}")
        task = await call(factory, dispatcher, create_text_message_object(
            Role.user, "Send a crew, it is dark near the clinic"))
        print(f"  state={task.status.state.value}  task={task.id}")
        print(f"  agent asks: {text_of(task)!r}")

        task2 = await call(factory, dispatcher, followup("Feeder F-114, clinic block", task))
        same = "SAME task" if task2.id == task.id else "NEW task - context lost"
        print(f"  state={task2.status.state.value}  {same}")
        print(f"  plan: {text_of(task2)!r}\n")

        notifier = await find_by_tag(hx, "comms")
        print(f"[ASK] tag=comms      -> {notifier.name} @ {notifier.url}")
        sms = await call(factory, notifier, create_text_message_object(
            Role.user, f"feeder F-114, {text_of(task2)}"))
        print(f"  sms: {text_of(sms)!r}")
        return text_of(sms)


FINAL_SMS = await outage_run()

# Production note: every send needs a timeout shorter than your own SLA, retries
# with backoff, and a trace ID in message metadata so a -32603 five hops away is
# diagnosable.

[ASK] tag=locate     -> Fault Locator @ http://127.0.0.1:9101/


  locator says: 'Maple Street is on feeder F-114.'

[ASK] tag=dispatch   -> Crew Dispatcher @ http://127.0.0.1:9102/
  state=input-required  task=2b9e50dd-7a6a-4e5d-b003-f5053c391347
  agent asks: 'Which feeder segment? Reply with the feeder ID, e.g. F-114.'
  state=completed  SAME task
  plan: 'Crew ALPHA-2, P1, ETA 45 min'

[ASK] tag=comms      -> Customer Notifier @ http://127.0.0.1:9103/
  sms: 'Aurora Grid: fault found on feeder F-114. Crew ALPHA-2 en route, power back by 15:05.'


## Stage 6: the four bugs, before and after

Every one returns HTTP 200 and a well formed result. None throws. That is why they pass review and die in production.

### Bug 1: orphaned task

Already proved on the wire above. Restated as the one line fix.

| | |
| --- | --- |
| Symptom | Turn 2 gets a different task id, state `completed`, no error |
| Cause | The follow-up message carries no `taskId` |
| Fix | `msg.task_id = task.id` and `msg.context_id = task.context_id` |

### Bug 2: shredded artifact

Artifact parts are **appended**, never separated. A streaming agent emits one part per chunk and the boundaries are arbitrary. A separator is a decision the sender never authorised.

In [22]:
async def bug_2():
    async with httpx.AsyncClient(timeout=120) as hx:
        from a2a.client import A2ACardResolver

        card = await A2ACardResolver(httpx_client=hx, base_url=BASE["locator"]).get_agent_card()
        factory = ClientFactory(ClientConfig(httpx_client=hx, streaming=False,
                                             supported_transports=[TransportProtocol.jsonrpc]))
        task = await call(factory, card, create_text_message_object(
            Role.user, "Lights out on Maple Street"))
        parts = [p.root.text for a in (task.artifacts or []) for p in a.parts
                 if isinstance(p.root, TextPart)]
        print("parts on the wire:", len(parts))
        print("first six        :", parts[:6])
        print()
        print('BROKEN  " ".join ->', repr(" ".join(parts)))
        print('FIXED    "".join ->', repr("".join(parts)))


await bug_2()

parts on the wire: 12
first six        : ['Map', 'le ', 'Str', 'eet', ' is', ' on']

BROKEN  " ".join -> 'Map le  Str eet  is  on  fe ede r F -11 4. '
FIXED    "".join -> 'Maple Street is on feeder F-114.'


### Bug 3: untagged skill

The agent is healthy, published, reachable, and invisible. Strands can derive a skill `id` and `description` from your function and its docstring. It cannot invent tags, and tags are the only field a capability query can filter on.

In [23]:
from strands.multiagent.a2a import A2AServer
from a2a.types import AgentSkill

broken = A2AServer(agent=locator_agent, host="127.0.0.1", port=9199, version="1.0.0")
fixed = A2AServer(
    agent=locator_agent, host="127.0.0.1", port=9199, version="1.0.0",
    skills=[AgentSkill(id="locate_fault", name="Locate fault",
                       description="Map an outage complaint to a feeder segment.",
                       tags=["outage", "locate"])],
)

for label, server in [("BROKEN (auto-derived)", broken), ("FIXED (declared)", fixed)]:
    card = server.public_agent_card
    skills = [(s.id, s.tags) for s in card.skills]
    matches = [s.id for s in card.skills if "locate" in s.tags]
    print(f"{label:<24} skills={skills}")
    print(f"{'':<24} tag=locate matches -> {matches or 'NOTHING'}")

BROKEN (auto-derived)    skills=[('lookup_feeder', [])]
                         tag=locate matches -> NOTHING
FIXED (declared)         skills=[('locate_fault', ['outage', 'locate'])]
                         tag=locate matches -> ['locate_fault']


### Bug 3b: the decorator that fails without failing

An undecorated function passed to `tools=` is dropped. Strands logs one line to stderr and carries on. The agent builds, the tool never fires, and the skill never reaches the card.

In [24]:
from strands import Agent, tool
from aurora_common import strands_model


def plain_function(x: str) -> str:
    """No decorator, so this is invisible."""
    return x


@tool
def decorated_tool(x: str) -> str:
    """Decorated, so this becomes a skill."""
    return x


probe = Agent(name="Probe", description="Decorator probe",
              model=strands_model("ok"), tools=[plain_function, decorated_tool])
print("skills on the card:", [(s.id, s.tags) for s in A2AServer(agent=probe, port=9198).agent_skills])
print("(the stderr line above is the ONLY signal that plain_function was dropped)")

tool=<<function plain_function at 0x7f3072f556c0>> | unrecognized tool specification


skills on the card: [('decorated_tool', [])]
(the stderr line above is the ONLY signal that plain_function was dropped)


### Bug 4: unreachable url

No code fix runs here. The card is valid, well formed, and lies to anyone off box.

| | |
| --- | --- |
| Who breaks | Every caller not on this host. A sibling container, a pod on another node, a partner in another data centre |
| Why it survives review | The local coordinator keeps working, so tests pass |
| The fix | `card.url` comes from configuration, not from the bind address. Strands exposes `http_url=` for this, plus `serve_at_root=True` for load balancers that strip path prefixes |
| The rule | The bind address is where the socket listens. `card.url` is where the world dials. Different facts, different variables |

In [25]:
import inspect

from strands.multiagent.a2a import A2AServer

sig = inspect.signature(A2AServer.__init__)
for name in ["host", "port", "http_url", "serve_at_root"]:
    p = sig.parameters[name]
    print(f"{name:<15} default={p.default!r}")

print()
local = A2AServer(agent=locator_agent, host="127.0.0.1", port=9197)
public = A2AServer(agent=locator_agent, host="127.0.0.1", port=9197,
                   http_url="https://agents.aurora-grid.example/locator")
print("bind-derived url :", local.public_agent_card.url)
print("config-driven url:", public.public_agent_card.url)

host            default='127.0.0.1'
port            default=9000
http_url        default=None
serve_at_root   default=False

bind-derived url : http://127.0.0.1:9197/
config-driven url: https://agents.aurora-grid.example/locator/


### Bug 5: the port collision you will hit for real

Strands `A2AServer` defaults to port **9000**. AgentCore Runtime is required to bind **9000**. Start both and the second dies with `address already in use`, or worse, the first silently takes the traffic.

This notebook uses 9100 to 9103 for exactly that reason.

When you diagnose it live, `lsof -i :9000 -sTCP:LISTEN` is the correct incantation. Without `-sTCP:LISTEN` you match your own process and kill your own kernel.

In [26]:
print("Strands A2AServer default port:", inspect.signature(A2AServer.__init__).parameters["port"].default)
print("Ports this notebook uses      :", PORTS)
print("Diagnosis command             : lsof -i :9000 -sTCP:LISTEN")

Strands A2AServer default port: 9000
Ports this notebook uses      : {'registry': 9100, 'locator': 9101, 'dispatcher': 9102, 'notifier': 9103}
Diagnosis command             : lsof -i :9000 -sTCP:LISTEN


### The streaming flag, measured

The Strands default is not A2A compliant. The SDK itself warns about it. The cost is `task.history`: one entry per stream fragment instead of one entry per turn.

The cell below starts two locators on separate ports, one with the flag and one without, and counts.

In [27]:
import textwrap

FLAG_PROBE = textwrap.dedent('''
    import sys
    from strands.multiagent.a2a import A2AServer
    from lab1_agents import locator_agent

    port = int(sys.argv[1])
    compliant = sys.argv[2] == "1"
    A2AServer(agent=locator_agent, host="127.0.0.1", port=port, version="1.0.0",
              enable_a2a_compliant_streaming=compliant).serve()
''')
(LAB / "flag_probe.py").write_text(FLAG_PROBE)

flag_procs = {}
for port, compliant in [(9111, "0"), (9112, "1")]:
    flag_procs[port] = subprocess.Popen(
        [sys.executable, "flag_probe.py", str(port), compliant],
        cwd=str(LAB), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for port in flag_procs:
    deadline = time.time() + 90
    while time.time() < deadline:
        try:
            if httpx.get(f"http://127.0.0.1:{port}/.well-known/agent-card.json", timeout=3).status_code == 200:
                break
        except Exception:
            time.sleep(1)

print(f"{'setting':<48} {'history':>8} {'artifact parts':>16}")
print("-" * 74)
for port, label in [(9111, "enable_a2a_compliant_streaming=False (default)"),
                    (9112, "enable_a2a_compliant_streaming=True")]:
    r = rpc("message/send", {"message": {"kind": "message", "messageId": f"f-{port}",
            "role": "user", "parts": [{"kind": "text", "text": "Lights out on Maple Street"}]}},
            req_id=f"flag-{port}", url=f"http://127.0.0.1:{port}/")["result"]
    parts = sum(len(a["parts"]) for a in r.get("artifacts", []))
    print(f"{label:<48} {len(r.get('history', [])):>8} {parts:>16}")

for proc in flag_procs.values():
    proc.terminate()

# Production note: assert on len(task.history) in CI, not just on the final text.
# A history polluted with fragments is replayed by any downstream agent that reads it.

setting                                           history   artifact parts
--------------------------------------------------------------------------
enable_a2a_compliant_streaming=False (default)         12                1
enable_a2a_compliant_streaming=True                     1               12


## Self check

One cell, every claim in this notebook verified against live state. Any `FAIL` means something drifted.

In [28]:
checks = []


def check(name, condition, detail=""):
    checks.append((name, bool(condition), detail))


card_locator = httpx.get(f"{BASE['locator']}/.well-known/agent-card.json", timeout=10).json()
card_dispatch = httpx.get(f"{BASE['dispatcher']}/.well-known/agent-card.json", timeout=10).json()

check("all four services alive", all(p.poll() is None for p in PROCS.values()))
check("card served on well known path", card_locator["name"] == "Fault Locator")
check("protocolVersion is 0.3.0", card_locator["protocolVersion"] == "0.3.0")
check("preferredTransport is JSONRPC", card_locator["preferredTransport"] == "JSONRPC")
check("skills carry tags", card_locator["skills"][0]["tags"] == ["outage", "locate"])
check("card url matches its port", card_dispatch["url"] == f"{BASE['dispatcher']}/")
check("retired card path still 200s",
      httpx.get(f"{BASE['locator']}/.well-known/agent.json", timeout=10).status_code == 200)

check("message/send returns a Task", turn1["result"]["kind"] == "task")
check("vague ask parks at input-required", turn1["result"]["status"]["state"] == "input-required")
check("taskId resumes the same task", with_ids["id"] == TASK_ID)
check("missing taskId orphans it", without["id"] != TASK_ID)
check("resumed task completed", with_ids["status"]["state"] == "completed")
check("task_id alone is sufficient", only_task["id"] == seed["id"])
check("server backfills contextId", only_task["contextId"] == seed["contextId"])
check("tasks/get finds the task", fetched["id"] == TASK_ID)
check("answer is in artifacts, not history",
      fetched.get("artifacts") and all(
          "ALPHA-2" not in " ".join(p["text"] for p in m["parts"])
          for m in fetched.get("history", [])))

check("bad method returns -32601", rpc("message/sned", {}).get("error", {}).get("code") == -32601)
check("unknown task returns -32001",
      rpc("tasks/get", {"id": "nope"}).get("error", {}).get("code") == -32001)
check("bad params returns -32602",
      rpc("message/send", {"message": {"role": "user"}}).get("error", {}).get("code") == -32602)
check("terminal task cannot be canceled",
      rpc("tasks/cancel", {"id": TASK_ID}).get("error", {}).get("code") == -32002)

check("registry ASK finds dispatcher",
      [c["name"] for c in httpx.get(f"{BASE['registry']}/agents",
                                    params={"tag": "dispatch"}).json()] == ["Crew Dispatcher"])
check("registry returns nothing for an unknown tag",
      httpx.get(f"{BASE['registry']}/agents", params={"tag": "billing"}).json() == [])
check("registry publishes its own card",
      httpx.get(f"{BASE['registry']}/.well-known/agent-card.json").json()["name"] == "Aurora Registry")

check("auto-derived skills have empty tags", broken.public_agent_card.skills[0].tags == [])
check("declared skills carry tags", fixed.public_agent_card.skills[0].tags == ["outage", "locate"])
check("http_url overrides the bind address",
      public.public_agent_card.url.startswith("https://agents.aurora-grid.example"))
check("full outage run produced an SMS", "ALPHA-2" in FINAL_SMS)

width = max(len(n) for n, _, _ in checks)
passed = sum(ok for _, ok, _ in checks)
for name, ok, detail in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {name:<{width}}  {detail}")
print()
print(f"{passed}/{len(checks)} checks passed")
assert passed == len(checks), "self check failed"

[PASS] all four services alive                      
[PASS] card served on well known path               
[PASS] protocolVersion is 0.3.0                     
[PASS] preferredTransport is JSONRPC                
[PASS] skills carry tags                            
[PASS] card url matches its port                    
[PASS] retired card path still 200s                 
[PASS] message/send returns a Task                  
[PASS] vague ask parks at input-required            
[PASS] taskId resumes the same task                 
[PASS] missing taskId orphans it                    
[PASS] resumed task completed                       
[PASS] task_id alone is sufficient                  
[PASS] server backfills contextId                   
[PASS] tasks/get finds the task                     
[PASS] answer is in artifacts, not history          
[PASS] bad method returns -32601                    
[PASS] unknown task returns -32001                  
[PASS] bad params returns -32602              

## Where this fails

An honest list. None of it is solved by this notebook.

| Gap | What actually happens | What you need on top |
| --- | --- | --- |
| **No registry standard** | Every vendor invents an incompatible query API. Your coordinator is coupled to your registry | Treat the registry client as a port with one adapter per environment |
| **Cards are self asserted** | Any agent can claim any skill. `tags` are marketing copy until someone verifies them | `AgentCardSignature` plus an issuer allowlist held by the registry |
| **`url` is a promise, not a fact** | Cards go stale, hosts move, load balancers rewrite paths. Nothing revalidates | Health check on refresh, short cache TTL, treat `url` as a hint |
| **No cost or quota in the protocol** | A2A has no field for price, rate limit, or token budget. A runaway coordinator fans out without limit | Enforce at your gateway, not in the agent |
| **No transaction semantics** | Dispatcher completes, Notifier fails, the crew is rolling and nobody was told. There is no rollback | Idempotency keys and a compensating action per side effect, in the coordinator |
| **Errors are shallow** | `-32603 Internal error` tells you nothing about which of five downstream agents broke | Trace ID propagated as message metadata, plus your own observability |
| **Streaming is inconsistent across SDKs** | The Strands default emits non compliant chunk messages until you flip a flag | Assert on `task.history` length in CI, not just on the final text |
| **`input-required` has no timeout** | A task can sit waiting for a human forever, holding a queue slot | Your own reaper, since the protocol will not do it |
| **`InMemoryTaskStore` loses everything on restart** | A rolling deploy drops every open task, including those parked on a human | `DatabaseTaskStore` with Postgres or MySQL |

## What changes in production

| Notebook | Production |
| --- | --- |
| `subprocess.Popen` | One container per agent, own scaling and rollout |
| `AURORA_MOCK=1` | Real Bedrock, `us.` inference profile prefix mandatory for Claude |
| Access keys or none | IAM roles, least privilege, no long lived credentials on the agent host |
| `url` built from `BASE[role]` | `http_url` from configuration, `serve_at_root` behind a path stripping load balancer |
| `InMemoryTaskStore` | `DatabaseTaskStore` |
| Bare `message/send` | Timeout shorter than your SLA, retries with backoff, trace ID in metadata |
| Registry with a seed list | Signed cards, issuer allowlist, health checked refresh, tenant scoping |
| Polling `tasks/get` | `tasks/pushNotificationConfig/set` plus an authenticated webhook with replay protection |

## One page summary

| Layer | Object | Carries | Verb that touches it |
| --- | --- | --- | --- |
| Discovery | Agent Card | WHO, WHERE, WHAT, HOW | `GET /.well-known/agent-card.json` |
| Conversation | Message | `role`, `parts[]`, `messageId` | `message/send` |
| Work | Task | `id`, `contextId`, `status.state`, `history[]` | `tasks/get`, `tasks/cancel` |
| Result | Artifact | `name`, `parts[]` appended | Read off a completed Task |
| Wire | JSON-RPC | `jsonrpc`, `id`, `method`, `params` | POST to the card's `url` |

**Three doors:** KNOCK a domain, ASK a registry, DIAL a config.
**Four exits:** completed, canceled, failed, rejected.
**Two waits:** input-required, auth-required.
**Two error bands:** minus 326xx is the envelope, minus 320xx is the ask.

In [29]:
stop_all()
for role, proc in PROCS.items():
    print(f"{role:<11} pid={proc.pid:<7} exit={proc.poll()}")
print("\nall services stopped")

locator     pid=601     exit=-15
dispatcher  pid=602     exit=-15
notifier    pid=603     exit=-15
registry    pid=604     exit=-15

all services stopped
